# 04 — Hard Benign Negatives

## Research Question
Is the detector learning adversarial behavior, or merely task/execution complexity?

## Hypothesis
If telemetry truly captures adversarial behavior, performance should remain meaningful under hard benign negatives matched by complexity.

## Input Data
- `episode_df_all`
- optionally `early_df_all`

## Prediction/Analysis Target
- `is_attack` under hard benign selection

## Validation Protocol
Construct hard benign subsets, then evaluate binary and per-attack detection with leave-one-seed-out.

## Expected Paper Artifact
- Hard benign dataset
- Detection degradation analysis
- Hard-benign ablation tables


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    make_scorer,
)

from evaluation import (
    load_jsonl, 
    evaluate_binary_prediction, 
    evaluate_multiclass_attack_type,
    fit_rf_feature_importance,
    evaluate_multiclass_feature_group_ablation,
    add_akc_phase_from_attack_type,
    normalize_seed_column,
    infer_telemetry_features,
    prepare_akc_phase_df,
    run_akc_phase_leave_one_seed,
    summarize_multiclass_results,
    collect_akc_phase_predictions_leave_one_seed,
    plot_confusion_matrix_from_predictions,
    infer_early_akc_features,
    classification_report,
    run_temporal_akc_phase_classification,
    plot_temporal_akc_curves,
    adversarial_fragmentation_index,
    risk_conditioned_fragmentation,
    semantic_fragmentation_proxy,
    add_benign_normalized_fragmentation,
    evaluate_leave_one_seed_out,
    TELEMETRY_NUMERIC,
    evaluate_feature_group_ablation,
    FEATURE_GROUPS,
    evaluate_leave_one_attack_out,
    compute_permutation_importance,
    run_operational_only_leave_one_seed,
    summarize_results,
    plot_exp1_operational_comparison,
    load_tamas_jsonl,
    build_early_df_from_agent_outputs,
    run_early_detection_leave_one_seed,
    plot_early_detection_curve,
    EARLY_FEATURES_CANDIDATES,
    validate_early_df_for_attack_analysis,
    run_early_detection_by_attack_type,
    summarize_early_detection_by_attack,
    plot_metric_curves_by_attack,
    compute_temporal_gain,
    infer_numeric_features,
    keep_existing_numeric,
    run_early_detection_feature_ablation,
    plot_ablation_curves,
    telemetry_guardrail_decision,
    run_akc_phase_leave_one_attack_type_out,
)
from datasets.tamas.tamas_features import build_all_feature_tables

RESULTS_DIR = Path("results/tamas")
RAW_DIR = RESULTS_DIR / "raw"
FEATURES_DIR = RESULTS_DIR / "features"
PLOTS_DIR = RESULTS_DIR / "plots"

scenario = "education"
architecture = "centralized_tamas"
CONDITIONS = [
    "benign",
    "byzantine",
    "colluding",
    "contradicting",
    "DPI",
    "impersonation",
    "IPI",
]
SEEDS = [1, 2, 3]
MODEL_NAMES = [
    "ticlazau/meta-llama-3.1-8b-instruct:latest",
    # "qwen3",
]

In [ ]:

# Load processed telemetry tables generated by 00_setup_and_feature_extraction.ipynb.
# If files are not available yet, run notebook 00 first.
PROCESSED_DIR = Path("results/tamas/processed")

for _name in ["episode_df_all", "agent_df_all", "early_df_all", "df_raw_all"]:
    _path = PROCESSED_DIR / f"{_name}.parquet"
    if _path.exists():
        globals()[_name] = pd.read_parquet(_path)
        print(f"Loaded {_name}: {globals()[_name].shape}")
    else:
        print(f"Missing {_path}; run 00_setup_and_feature_extraction.ipynb if this notebook needs it.")


### Experimento 2 — Hard benign negatives

Objetivo: testar se o modelo está aprendendo risco real ou apenas “complexidade do workflow”.

In [ ]:
# exp2_results, hard_benign_df = run_hard_benign_leave_one_seed(
#     df=episode_df_all,
#     quantile=0.70,
# )

# print(exp2_results)

# exp2_summary = summarize_results(
#     exp2_results,
#     group_cols=["feature_set", "model", "hard_benign_quantile"],
# )
# print(exp2_summary)

In [ ]:
# # build_hard_benign_dataset adiciona complexity_score internamente à cópia, então vamos criar explicitamente:
# hard_benign_df_for_plot = build_hard_benign_dataset(episode_df_all, quantile=0.70)

# # Para plotar, precisamos também adicionar complexity_score ao original:
# episode_df_with_complexity = build_hard_benign_dataset(episode_df_all, quantile=0.0)

# plot_complexity_distribution(
#     df=episode_df_with_complexity,
#     hard_df=hard_benign_df_for_plot,
# )

In [ ]:
# hard_early_df = build_hard_benign_early_df(
#     early_df_all,
#     quantile=0.70,
# )

# print(hard_early_df.shape)
# print(hard_early_df["is_attack"].value_counts())
# print(hard_early_df["trace_fraction"].value_counts().sort_index())

### Early detection por ataque usando hard benign

In [ ]:
# hard_attack_results = run_early_detection_by_attack_type_safe(
#     early_df=hard_early_df,
#     features=infer_numeric_features(hard_early_df),
#     model_kind="rf",
#     experiment_name="hard_benign_by_attack_all_features",
# )

# hard_attack_summary = summarize_early_detection_by_attack(hard_attack_results)

# hard_attack_summary

### Hard benign + ablação de feature sets

In [ ]:
# hard_attack_ablation_results = run_hard_benign_attack_feature_ablation(
#     hard_early_df=hard_early_df,
#     feature_sets=FEATURE_SETS_ABLATION,
#     model_kind="rf",
# )

# hard_attack_ablation_summary = summarize_results(
#     hard_attack_ablation_results,
#     group_cols=["feature_set", "attack_type", "trace_fraction"],
# )

# hard_attack_ablation_summary

In [ ]:
# plot_attack_fraction_heatmap_from_summary(
#     hard_attack_summary,
#     metric="roc_auc_mean",
#     output_path=OUTPUT_DIR / "heatmap_hard_benign_by_attack_roc_auc.png",
#     title="Hard Benign Early Detection by Attack Type: ROC-AUC",
# )

In [ ]:
# plot_attack_fraction_heatmap_from_summary(
#     hard_attack_ablation_summary,
#     metric="roc_auc_mean",
#     feature_set="operational_cost_latency_tokens_only",
#     output_path=OUTPUT_DIR / "heatmap_hard_benign_operational_only_roc_auc.png",
#     title="Hard Benign + Operational-only: ROC-AUC",
# )

In [ ]:
# comparison_75 = compare_feature_sets_at_fraction(
#     hard_attack_ablation_summary,
#     fraction=0.75,
#     metric="roc_auc_mean",
# )

# comparison_75

In [ ]:
# timing_ablation = classify_detection_timing_grouped(
#     hard_attack_ablation_summary,
#     metric="roc_auc_mean",
#     threshold=0.85,
# )

# timing_ablation